**Persistent Cell IDs:**

Each cell gets a unique global ID that persists throughout its entire lifecycle
Centroid-Based Tracking: Uses cell center positions to track movement between
frames

**Hungarian Algorithm:** Optimal assignment of cells between consecutive frames

**New Cell Detection:** Automatically detects when cells divide or appear

**Data:** Consistent IDs perfect for Fourier analysis



**Files for the Analysis:**

consistent_cell_boundaries.csv - Boundary data with persistent cell IDs
consistent_shape_summary.csv - Shape metrics with consistent tracking
cell_trajectories.csv - Movement patterns for each cell
detailed_tracking_info.csv - Complete tracking information

**Visualization Files:**

consistently_tracked_stack.tif - Stack with consistent IDs

consistently_tracked_stack_rgb.tif - Color-coded visualization

consistently_tracked_stack_with_ids.tif - With ID numbers overlaid

**Parameters to adjust**

max_distance=50: Maximum pixels a cell can move between frames


min_area=50: Minimum cell size to track

max_gap=5: Remove cells missing for 5+ consecutive frames



In [ ]:
import numpy as np
import pandas as pd
import tifffile
from pathlib import Path
from skimage import measure, segmentation
import matplotlib.pyplot as plt
from matplotlib import colors
import cv2
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment

def track_cells_across_frames(stack_file, max_distance=50, min_area=50):
    """
    Track cells across frames and assign consistent IDs

    Args:
        stack_file: Path to segmentation stack
        max_distance: Maximum distance between centroids to consider same cell
        min_area: Minimum area to consider a valid cell

    Returns:
        tracked_stack: Stack with consistent cell IDs
        tracking_info: DataFrame with tracking information
    """

    print("=== CELL TRACKING ACROSS FRAMES ===")
    print(f"Loading stack: {stack_file}")

    # Load segmentation stack
    mask_stack = tifffile.imread(stack_file)
    n_timepoints, height, width = mask_stack.shape

    print(f"Stack shape: {mask_stack.shape}")
    print(f"Processing {n_timepoints} timepoints...")

    # Initialize tracking
    tracked_stack = np.zeros_like(mask_stack, dtype=np.uint32)
    next_global_id = 1
    active_cells = {}  # global_id -> {centroid, area, last_seen}
    tracking_records = []

    print("-" * 60)

    for t in range(n_timepoints):
        print(f"Frame {t}: ", end="")

        frame = mask_stack[t]
        current_cells = {}

        # Extract cell properties from current frame
        for region in measure.regionprops(frame):
            if region.area >= min_area:
                current_cells[region.label] = {
                    'centroid': region.centroid,
                    'area': region.area,
                    'bbox': region.bbox
                }

        print(f"{len(current_cells)} cells detected", end="")

        if t == 0:
            # First frame - assign initial IDs
            for orig_id, props in current_cells.items():
                active_cells[next_global_id] = {
                    'centroid': props['centroid'],
                    'area': props['area'],
                    'last_seen': t
                }
                tracked_stack[t][frame == orig_id] = next_global_id

                tracking_records.append({
                    'timepoint': t,
                    'global_id': next_global_id,
                    'original_id': orig_id,
                    'centroid_y': props['centroid'][0],
                    'centroid_x': props['centroid'][1],
                    'area': props['area'],
                    'status': 'new'
                })

                next_global_id += 1

            print(f" -> assigned IDs 1-{next_global_id-1}")

        else:
            # Track cells from previous frames
            if not current_cells:
                print(" -> no cells found")
                continue

            # Calculate distances between current and active cells
            current_centroids = np.array([props['centroid'] for props in current_cells.values()])
            current_ids = list(current_cells.keys())

            active_ids = list(active_cells.keys())
            if active_ids:
                active_centroids = np.array([active_cells[gid]['centroid'] for gid in active_ids])

                # Compute distance matrix
                distances = cdist(current_centroids, active_centroids)

                # Solve assignment problem
                current_indices, active_indices = linear_sum_assignment(distances)

                assigned_current = set()
                assigned_active = set()

                # Process valid assignments
                for c_idx, a_idx in zip(current_indices, active_indices):
                    if distances[c_idx, a_idx] <= max_distance:
                        orig_id = current_ids[c_idx]
                        global_id = active_ids[a_idx]

                        # Update active cell info
                        active_cells[global_id] = {
                            'centroid': current_cells[orig_id]['centroid'],
                            'area': current_cells[orig_id]['area'],
                            'last_seen': t
                        }

                        # Assign in tracked stack
                        tracked_stack[t][frame == orig_id] = global_id

                        tracking_records.append({
                            'timepoint': t,
                            'global_id': global_id,
                            'original_id': orig_id,
                            'centroid_y': current_cells[orig_id]['centroid'][0],
                            'centroid_x': current_cells[orig_id]['centroid'][1],
                            'area': current_cells[orig_id]['area'],
                            'status': 'tracked'
                        })

                        assigned_current.add(c_idx)
                        assigned_active.add(a_idx)

                # Handle unassigned current cells (new cells)
                for c_idx, orig_id in enumerate(current_ids):
                    if c_idx not in assigned_current:
                        active_cells[next_global_id] = {
                            'centroid': current_cells[orig_id]['centroid'],
                            'area': current_cells[orig_id]['area'],
                            'last_seen': t
                        }

                        tracked_stack[t][frame == orig_id] = next_global_id

                        tracking_records.append({
                            'timepoint': t,
                            'global_id': next_global_id,
                            'original_id': orig_id,
                            'centroid_y': current_cells[orig_id]['centroid'][0],
                            'centroid_x': current_cells[orig_id]['centroid'][1],
                            'area': current_cells[orig_id]['area'],
                            'status': 'new'
                        })

                        next_global_id += 1

                tracked_count = len(assigned_current)
                new_count = len(current_cells) - tracked_count

            else:
                # No active cells - all are new
                tracked_count = 0
                new_count = len(current_cells)

                for orig_id, props in current_cells.items():
                    active_cells[next_global_id] = {
                        'centroid': props['centroid'],
                        'area': props['area'],
                        'last_seen': t
                    }

                    tracked_stack[t][frame == orig_id] = next_global_id

                    tracking_records.append({
                        'timepoint': t,
                        'global_id': next_global_id,
                        'original_id': orig_id,
                        'centroid_y': props['centroid'][0],
                        'centroid_x': props['centroid'][1],
                        'area': props['area'],
                        'status': 'new'
                    })

                    next_global_id += 1

            print(f" -> {tracked_count} tracked, {new_count} new")

        # Remove cells not seen for too long (optional - adjust threshold)
        max_gap = 5  # Remove cells not seen for 5+ frames
        to_remove = []
        for gid, info in active_cells.items():
            if t - info['last_seen'] > max_gap:
                to_remove.append(gid)

        for gid in to_remove:
            del active_cells[gid]

    print("-" * 60)
    print(f"Tracking completed!")
    print(f"Total unique cells tracked: {next_global_id - 1}")
    print(f"Total tracking records: {len(tracking_records)}")

    # Create tracking DataFrame
    tracking_df = pd.DataFrame(tracking_records)

    # Summary statistics
    cell_lifespans = tracking_df.groupby('global_id')['timepoint'].agg(['min', 'max', 'count'])
    cell_lifespans['lifespan'] = cell_lifespans['max'] - cell_lifespans['min'] + 1

    print(f"Average cell lifespan: {cell_lifespans['lifespan'].mean():.1f} frames")
    print(f"Longest-lived cell: {cell_lifespans['lifespan'].max()} frames")
    print(f"Shortest-lived cell: {cell_lifespans['lifespan'].min()} frames")

    return tracked_stack, tracking_df, cell_lifespans

def create_tracking_visualizations(tracked_stack, tracking_df, output_folder):
    """
    Create visualizations of the tracked cells
    """
    output_path = Path(output_folder)
    output_path.mkdir(parents=True, exist_ok=True)

    print("\n=== CREATING TRACKING VISUALIZATIONS ===")

    # 1. Save tracked stack
    stack_file = output_path / "consistently_tracked_stack.tif"
    tifffile.imwrite(stack_file, tracked_stack,
                    photometric='minisblack',
                    metadata={'axes': 'TYX'})
    print(f"✓ Saved tracked stack: {stack_file}")

    # 2. Create RGB visualization with consistent colors
    n_timepoints, height, width = tracked_stack.shape
    rgb_stack = np.zeros((n_timepoints, height, width, 3), dtype=np.uint8)

    # Get all unique global IDs
    all_ids = np.unique(tracked_stack[tracked_stack > 0])
    n_colors = len(all_ids)

    # Create consistent colormap
    colors_list = plt.cm.get_cmap('tab20', n_colors)
    id_to_color = {}
    for i, cell_id in enumerate(all_ids):
        color = colors_list(i / max(1, n_colors-1))
        id_to_color[cell_id] = (np.array(color[:3]) * 255).astype(np.uint8)

    for t in range(n_timepoints):
        frame = tracked_stack[t]
        rgb_frame = np.zeros((height, width, 3), dtype=np.uint8)

        for cell_id in np.unique(frame)[1:]:
            if cell_id in id_to_color:
                mask = frame == cell_id
                rgb_frame[mask] = id_to_color[cell_id]

        rgb_stack[t] = rgb_frame

    rgb_file = output_path / "consistently_tracked_stack_rgb.tif"
    tifffile.imwrite(rgb_file, rgb_stack,
                    photometric='rgb',
                    metadata={'axes': 'TYXC'})
    print(f"✓ Saved RGB visualization: {rgb_file}")

    # 3. Create annotated version with cell IDs
    annotated_stack = np.zeros_like(rgb_stack)

    for t in range(n_timepoints):
        frame = tracked_stack[t]
        rgb_frame = rgb_stack[t].copy()

        for cell_id in np.unique(frame)[1:]:
            cell_mask = frame == cell_id
            props = measure.regionprops(cell_mask.astype(int))

            if props:
                centroid = props[0].centroid
                y, x = int(centroid[0]), int(centroid[1])

                # Add cell ID as text
                text = str(int(cell_id))
                cv2.putText(rgb_frame, text, (x-10, y+5),
                          cv2.FONT_HERSHEY_SIMPLEX, 0.5,
                          (255, 255, 255), 1, cv2.LINE_AA)  # White text

        annotated_stack[t] = rgb_frame

    annotated_file = output_path / "consistently_tracked_stack_with_ids.tif"
    tifffile.imwrite(annotated_file, annotated_stack,
                    photometric='rgb',
                    metadata={'axes': 'TYXC'})
    print(f"✓ Saved annotated visualization: {annotated_file}")

    return stack_file, rgb_file, annotated_file

def extract_consistent_boundaries_and_shape(tracked_stack, output_folder):
    """
    Extract boundaries and shape metrics using consistent cell IDs
    """
    print("\n=== EXTRACTING CONSISTENT BOUNDARIES ===")

    n_timepoints = tracked_stack.shape[0]
    all_boundaries = []
    shape_summary = []

    for t in range(n_timepoints):
        print(f"Processing timepoint {t+1}/{n_timepoints}")

        mask_frame = tracked_stack[t]

        for cell_id in np.unique(mask_frame)[1:]:  # Skip background
            cell_mask = (mask_frame == cell_id)

            # Get boundaries
            contours = measure.find_contours(cell_mask, 0.5)
            if len(contours) > 0:
                boundary = max(contours, key=len)  # Largest contour

                # Store boundary points
                boundary_coords = []
                for i, (y, x) in enumerate(boundary):
                    all_boundaries.append({
                        'time_point': t,
                        'cell_id': cell_id,  # This is now consistent across frames!
                        'boundary_point': i,
                        'x_coord': x,
                        'y_coord': y
                    })
                    boundary_coords.append([x, y])

                # Calculate shape metrics
                props = measure.regionprops(cell_mask.astype(int))[0]
                shape_summary.append({
                    'time_point': t,
                    'cell_id': cell_id,  # Consistent ID!
                    'centroid_x': props.centroid[1],
                    'centroid_y': props.centroid[0],
                    'area': props.area,
                    'perimeter': props.perimeter,
                    'eccentricity': props.eccentricity,
                    'solidity': props.solidity,
                    'n_boundary_points': len(boundary_coords)
                })

    # Save consistent tracking data
    output_path = Path(output_folder)

    boundaries_df = pd.DataFrame(all_boundaries)
    boundaries_file = output_path / "consistent_cell_boundaries.csv"
    boundaries_df.to_csv(boundaries_file, index=False)

    shape_df = pd.DataFrame(shape_summary)
    shape_file = output_path / "consistent_shape_summary.csv"
    shape_df.to_csv(shape_file, index=False)

    print(f"✓ Saved consistent boundaries: {boundaries_file}")
    print(f"✓ Saved shape summary: {shape_file}")
    print(f"Total boundary points: {len(boundaries_df)}")
    print(f"Unique cells tracked: {len(boundaries_df['cell_id'].unique())}")

    return boundaries_df, shape_df

def analyze_cell_trajectories(tracking_df, output_folder):
    """
    Analyze cell trajectories and create trajectory plots
    """
    print("\n=== ANALYZING CELL TRAJECTORIES ===")

    output_path = Path(output_folder)

    # Calculate trajectories
    trajectories = []
    for cell_id in tracking_df['global_id'].unique():
        cell_data = tracking_df[tracking_df['global_id'] == cell_id].sort_values('timepoint')

        if len(cell_data) >= 5:  # Only cells tracked for at least 5 frames
            trajectory = {
                'cell_id': cell_id,
                'first_frame': cell_data['timepoint'].min(),
                'last_frame': cell_data['timepoint'].max(),
                'lifespan': len(cell_data),
                'total_distance': 0,
                'avg_speed': 0
            }

            # Calculate movement
            coords = cell_data[['centroid_x', 'centroid_y']].values
            if len(coords) > 1:
                distances = np.sqrt(np.sum(np.diff(coords, axis=0)**2, axis=1))
                trajectory['total_distance'] = distances.sum()
                trajectory['avg_speed'] = distances.mean()

            trajectories.append(trajectory)

    traj_df = pd.DataFrame(trajectories)
    traj_file = output_path / "cell_trajectories.csv"
    traj_df.to_csv(traj_file, index=False)

    print(f"✓ Saved trajectories: {traj_file}")
    print(f"Cells with good trajectories (≥5 frames): {len(traj_df)}")

    return traj_df

def complete_cell_tracking_pipeline(stack_file, output_folder, max_distance=50):
    """
    Complete pipeline for consistent cell tracking and analysis
    """
    print("🔬 COMPLETE CELL TRACKING PIPELINE 🔬")
    print(f"Input: {stack_file}")
    print(f"Output: {output_folder}")
    print("="*70)

    # Step 1: Track cells across frames
    tracked_stack, tracking_df, lifespans = track_cells_across_frames(
        stack_file, max_distance=max_distance
    )

    # Step 2: Create visualizations
    create_tracking_visualizations(tracked_stack, tracking_df, output_folder)

    # Step 3: Extract consistent boundaries and shapes
    boundaries_df, shape_df = extract_consistent_boundaries_and_shape(
        tracked_stack, output_folder
    )

    # Step 4: Analyze trajectories
    trajectories_df = analyze_cell_trajectories(tracking_df, output_folder)

    # Step 5: Save tracking info
    output_path = Path(output_folder)
    tracking_df.to_csv(output_path / "detailed_tracking_info.csv", index=False)
    lifespans.to_csv(output_path / "cell_lifespans.csv")

    print("="*70)
    print("🎯 PIPELINE COMPLETE!")
    print(f"✓ Tracked {len(tracking_df['global_id'].unique())} unique cells")
    print(f"✓ Generated consistent boundary data for mathematician")
    print(f"✓ Ready for Fourier analysis with persistent cell IDs")
    print(f"✓ All files saved to: {output_folder}")

    return {
        'tracked_stack': tracked_stack,
        'tracking_df': tracking_df,
        'boundaries_df': boundaries_df,
        'shape_df': shape_df,
        'trajectories_df': trajectories_df
    }

# Usage:
"""
# Run complete tracking pipeline
results = complete_cell_tracking_pipeline(
    stack_file="/path/to/your/Stack_final_final.tif",
    output_folder="/path/to/output/",
    max_distance=50  # Adjust based on how far cells can move between frames
)

# Outputs:
# - consistent_cell_boundaries.csv: Boundaries with persistent cell IDs
# - consistent_shape_summary.csv: Shape metrics with persistent IDs
# - consistently_tracked_stack.tif: Visual verification
# - cell_trajectories.csv: Movement analysis
"""

In [ ]:
# Run the complete pipeline on your data
results = complete_cell_tracking_pipeline(
    stack_file="/content/drive/MyDrive/Colab Notebooks/test_model_cpsam_pupa/final_final/Stack_final_final.tif",
    output_folder="/content/drive/MyDrive/Colab Notebooks/test_model_cpsam_pupa/final_final/tracked_output/",
    max_distance=50  # Adjust if cells move very fast/slow between frames
)